In [1]:
import nest_asyncio
nest_asyncio.apply()
# 중첩 허용 : 이미 돌고 있는 루프 안에서 또 루프를 돌려도 괜찮게 규칙을 완화

## 비동기(async)란?

- 동기 (sync) : 일 하나가 끝나야 다음 일을 시작. ex) 순서대로 줄 서기
- 비동기 (async) : 기다리는 동안 다른 일 처리. ex) 주문하고 진동벨 받고 다른 일 하기

### asyncio

- 파이썬에서 비동기를 담당하는 엔진
- 이 엔진은 한 프로세스에 하나만 실행될 수 있다.

- 간혹 ipynb에서 jupiter가 이미 자체적으로 진행하고 있는 경우가 많음.
- 라마인덱스로 코드 실행 시 또 비동기를 진행시킬 수 있으므로 충돌이 나거나 오래 걸릴 수 있다.

# 데이터 로드

- RAG에 활용할 여러가지 확장자의 문서를 불러오는 방법들을 공부할 예정이다.
- 불러온 문서를 노드로 세분화 하는 방법

## 1. Doucument 객체 직접 생성

- 입력받은 텍스트, 문서 고유 id, 몇 가지 등등의 메타데이터(데이터에 대한 정보)를 딕셔너리 형태로 가지고 있다.

In [2]:
from llama_index.core import Document # 라마인덱스에서 가장 기본적인 문서 생성 방법

# 예시) 제품 설명
text = '''
AI 챗봇 사용 가이드:
- 간단한 질문으로 시작하세요.
- 구체적인 예시를 포함하면 더 좋은 답변을 받을 수 있습니다.
- 부적절한 내용은 자동으로 필터링됩니다.
'''

document = Document(
    text=text,
    metadata={'author':'제품개발팀', 'category':'사용설명서', 'version':'2.0'},
    id_='manual_ai_001',
)

print(document)

Doc ID: manual_ai_001
Text: AI 챗봇 사용 가이드: - 간단한 질문으로 시작하세요. - 구체적인 예시를 포함하면 더 좋은 답변을 받을 수
있습니다. - 부적절한 내용은 자동으로 필터링됩니다.


## 2. 기존 문서 로드하여 Document 객체 생성

In [3]:
from llama_index.core import SimpleDirectoryReader

# pdf 파일을 불러오기 및 documents 변수에 저장
documents = SimpleDirectoryReader(input_files=['data/pdf_example.pdf']).load_data()

In [4]:
# documents의 타입(자료형)
print(type(documents))

<class 'list'>


In [5]:
# documents의 내용
print(documents)

[Document(id_='cc0f5979-b9f9-4477-984d-b005657b5d36', embedding=None, metadata={'file_path': 'data\\pdf_example.pdf', 'file_name': 'pdf_example.pdf', 'file_type': 'application/pdf', 'file_size': 68851, 'creation_date': '2026-05-28', 'last_modified_date': '2026-06-03'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='%PDF-1.7\r\n%\r\n1 0 obj\r\n<</Type/Catalog/Pages 2 0 R/Lang(ko) /StructTreeRoot 15 0 R/MarkInfo<</Marked true>>/Metadata 292 0 R/ViewerPreferences 293 0 R>>\r\nendobj\r\n2 0 obj\r\n<</Type/Pages/Count 1/Kids[ 3 0 R] >>\r\nendobj\r\n3 0 obj\r\n<</Type/Page/Parent 2 0 R/Resources<</Font<</F1 5 0 R/F2 12 0 R>>/ExtGState<</GS10 10

In [6]:
print(documents[0]) # 실제 내용을 보려고

Doc ID: cc0f5979-b9f9-4477-984d-b005657b5d36
Text: %PDF-1.7  %  1 0 obj  <</Type/Catalog/Pages 2 0 R/Lang(ko)
/StructTreeRoot 15 0 R/MarkInfo<</Marked true>>/Metadata 292 0
R/ViewerPreferences 293 0 R>>  endobj  2 0 obj  <</Type/Pages/Count
1/Kids[ 3 0 R] >>  endobj  3 0 obj  <</Type/Page/Parent 2 0
R/Resources<</Font<</F1 5 0 R/F2 12 0 R>>/ExtGState<</GS10 10 0 R/GS11
11 0 R>>/ProcSet[/PDF/Text...


In [7]:
document = documents[0] # 실제 읽은 내용을 변수에 저장

# 문서 내용 확인
print(document.text)

%PDF-1.7
%
1 0 obj
<</Type/Catalog/Pages 2 0 R/Lang(ko) /StructTreeRoot 15 0 R/MarkInfo<</Marked true>>/Metadata 292 0 R/ViewerPreferences 293 0 R>>
endobj
2 0 obj
<</Type/Pages/Count 1/Kids[ 3 0 R] >>
endobj
3 0 obj
<</Type/Page/Parent 2 0 R/Resources<</Font<</F1 5 0 R/F2 12 0 R>>/ExtGState<</GS10 10 0 R/GS11 11 0 R>>/ProcSet[/PDF/Text/ImageB/ImageC/ImageI] >>/MediaBox[ 0 0 595.32 841.92] /Contents 4 0 R/Group<</Type/Group/S/Transparency/CS/DeviceRGB>>/Tabs/S/StructParents 0>>
endobj
4 0 obj
<</Filter/FlateDecode/Length 6428>>
stream
w-}:G_Bk?KnR }j)i7~?`RI}yARu): yH8Ï?_?_r>Onq_)!ָ0JKϟN?_\8?w?dZC}~>Տѕ_?O·÷6a~矞hs
/&u~u~l7%E8"-^whXax\^j:?xj/%RǶU%ZRѵQSYn\}Sv
p!n _C~!sﾱQou(ߵ5Kx]ɠ˵.YG>ZRvVrtʕ,-%'CPH&yv#yyrĆc^2
$ ׌cr.|^귋ٸQvEEkoAm~lG	;="s(ܘ"r8ɖ=rz%(MwP\(mLq=ܙo%S{QN\Y5.\c4$(F!Jf,ǶŖ"Ǉ<0a]K\ܟ(C-I$X=sߛdqXeZo,r{})[CnJ{`Z$pH{+]}!/-
d0L'r&ZƮ;FS7r$ŵ4)k0E_ax0me߇
9EC74)>nǶ}(uc>m+\pTV`+{R|褩ϵE`9-0G@(o'WJ?gC(auH~9Q

In [8]:
# 파일 정보 확인
print(document.metadata['file_name'])
print(document.metadata['file_type'])

pdf_example.pdf
application/pdf


In [9]:
# 문서 id 확인
print(document.id_)

cc0f5979-b9f9-4477-984d-b005657b5d36


## 3. 노드

- 라마인덱스에서 문서를 작은 단위로 나눈 기본 데이터 구조
- 검색과 질의응답을 효율적으로 수행할 수 있도록 도와준다.
- 특정 질문에 대해 관련성이 높은 부분만 검색하여 LLM이 보다 정확한 응답을 생성할 수 있다.

## 4. Node 객체 직접 생성

In [10]:
from llama_index.core import Document
from llama_index.core.schema import TextNode

document = Document(
    text='''인공지능은 우리의 미래를 변화시킬 것입니다.
            이러한 변화에 우리는 준비되어 있어야 합니다.'''
)

# 문서를 두 개의 노드로 분할 (각 문장을 하나의 노드로)
node1 = TextNode(text=document.text[:24], doc_id=document.id_)
node2 = TextNode(text=document.text[25:], doc_id=document.id_)

print(node1)
print(node2)

Node ID: c26112a7-dc0a-45ea-b137-88215d5e2e49
Text: 인공지능은 우리의 미래를 변화시킬 것입니다.
Node ID: 127d6591-8699-47e7-9040-803da09ff5b4
Text: 이러한 변화에 우리는 준비되어 있어야 합니다.


## 5. 문서를 불러와서 Document와 Node 객체 생성

- 노드파서 클래스 (SimpleNodeParser) : Document를 노드로 나눌때 사용되는 클래스
    - 파서 (Parser) : 큰 정보를 의미 있는 조각들로 나누는 것

In [11]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SimpleNodeParser

# docx 파일 불러오기 및 documnets 변수에 저장 (data폴더 안 docx확장자 파일 불러오기)
documents = SimpleDirectoryReader(input_dir='data', required_exts=['.docx']).load_data()

documents

[Document(id_='4f4ce76f-c097-47c1-a1b6-1f2650a2ada9', embedding=None, metadata={'file_path': 'c:\\Users\\jangc\\ai_class\\llm_labs\\data\\word_example.docx', 'file_name': 'word_example.docx', 'file_type': 'application/haansoftdocx', 'file_size': 15420, 'creation_date': '2026-05-28', 'last_modified_date': '2026-06-03'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x00ߤlZ\x01\x00\x00 \x05\x00\x00\x13\x00\x08\x02[Content_Types].xml \x04\x02(\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

- 청크 (chunk) : 파서를 통해 나누어진 조각, 노드와 비슷한 의미
- 

In [12]:
parser = SimpleNodeParser.from_defaults(
    chunk_size=200, # 200토큰을 기준으로 문서의 내용을 나눈다.
    chunk_overlap=20 # 20토큰씩 중복해서 겹쳐라.
)

nodes = parser.get_nodes_from_documents([documents[0]])

In [13]:
nodes

[TextNode(id_='c1e6bce8-a0b6-4c32-a784-59e993abcb11', embedding=None, metadata={'file_path': 'c:\\Users\\jangc\\ai_class\\llm_labs\\data\\word_example.docx', 'file_name': 'word_example.docx', 'file_type': 'application/haansoftdocx', 'file_size': 15420, 'creation_date': '2026-05-28', 'last_modified_date': '2026-06-03'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='4f4ce76f-c097-47c1-a1b6-1f2650a2ada9', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\jangc\\ai_class\\llm_labs\\data\\word_example.docx', 'file_name': 'word_example.docx', 'file_type': 'application/haansoftdocx', 'file_size': 15420, 'creation_date': '2026-05-28', 'last_modified_date': '2026-06-03'}, hash='12d5457f8c1fe

In [14]:
# 노드 확인
for i, node in enumerate(nodes):
    print(f'=== Node {i} ===')
    print(f'Text : {node.text}')
    print()

=== Node 0 ===
Text : PK    ! ߤlZ     [Content_Types].xml (                                                                                                                                          

=== Node 1 ===
Text :                                                                                                                                                                                  

=== Node 2 ===
Text :                                                                                                                                                                                  

=== Node 3 ===
Text :                                                                                  n0EUb袪*>-R{VǼQU
l"%33Vƃښl	w%=^i7+-d&0A6l4L60#ÒS
OX *V$z33%p)O^5}nH"

=== Node 4 ===
Text : 33%p)O^5}nH"dsXgL`|ԟ|Prۃs?PWtt4Q+"wa|T\y,NU%-D/ܚXݞ(<E) ;NL?F˼܉<Fk	hyڜqi?ޯli 1]Hgm@m   PK    !    N  _rels/.rels (             

In [15]:
# 메타데이터 확인
print(node.metadata)

{'file_path': 'c:\\Users\\jangc\\ai_class\\llm_labs\\data\\word_example.docx', 'file_name': 'word_example.docx', 'file_type': 'application/haansoftdocx', 'file_size': 15420, 'creation_date': '2026-05-28', 'last_modified_date': '2026-06-03'}


## 6. 특정 폴더(디렉토리) 안의 모든 파일 불러오기(로드)

In [16]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader('./data').load_data()

In [17]:
# 디렉토리 내 파일 개수
len(documents) # 폴더는 나오지 않는다

10

In [18]:
for d in documents:
    print(d.metadata['file_name'])

csv_example.csv
excel_example.xlsx
html_example.html
img_example.jpg
json_example.json
markdown_example.md
pdf_example.pdf
txt_example.txt
word_example.docx
xml_example.xml


In [19]:
# 하위 폴더를 포함
documents = SimpleDirectoryReader(input_dir='./data', recursive=True).load_data()

# 하위 폴더 포함 파일 개수
len(documents)

17

In [20]:
for d in documents:
    print(d.metadata['file_name'])

data_level0.bin
header.bin
length.bin
link_lists.bin
chroma.sqlite3
csv_example.csv
excel_example.xlsx
html_example.html
img_example.jpg
json_example.json
markdown_example.md
pdf_example.pdf
240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf
NIPS-2017-attention-is-all-you-need-Paper.pdf
txt_example.txt
word_example.docx
xml_example.xml


In [21]:
# 특정 파일만 불러오기
documents = SimpleDirectoryReader(input_dir='./data', input_files=['./data/json_example.json', './data/txt_example.txt'])